# Notebook 1 — Descriptive Statistics
**Real-Time E-commerce Price Intelligence Platform**

Covers: central tendency · dispersion · distribution shape · category distributions · time-series trends · volatility

---

In [ ]:
# papermill parameters
run_date = "2024-01-01"
output_html = "reports/01_descriptive_stats.html"

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings('ignore')

# Resolve project root whether running from notebooks/ or repo root
NB_DIR = Path().resolve()
ROOT = NB_DIR.parent.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(ROOT))

REPORTS_DIR = ROOT / 'analytics' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Run date  : {run_date}')
print(f'Root      : {ROOT}')
print(f'NumPy     : {np.__version__}')
print(f'Pandas    : {pd.__version__}')

## 1. Data Loading
Priority: Bigtable emulator → BigQuery mart → local JSONL → synthetic demo data

In [ ]:
def _load_jsonl() -> pd.DataFrame:
    frames = []
    for jsonl in (ROOT / 'data').rglob('*.jsonl'):
        try:
            frames.append(pd.read_json(jsonl, lines=True))
        except Exception:
            pass
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _synthetic() -> pd.DataFrame:
    """Reproducible demo data mirroring real scraper output schema."""
    rng = np.random.default_rng(42)
    sources = {
        'books_toscrape': dict(mu=15,  sigma=8,  n=400,  cats=['Fiction','Mystery','Science','History','Children']),
        'scrapeme_live':  dict(mu=30,  sigma=12, n=300,  cats=['Electronics','Toys','Clothing','Sports']),
        'jumia_ma':        dict(mu=250, sigma=80, n=350,  cats=['Phones','Computers','Home','Fashion','Beauty']),
        'ultrapc_ma':      dict(mu=450, sigma=150,n=200,  cats=['Laptops','Desktops','Monitors','Accessories']),
        'micromagma_ma':   dict(mu=380, sigma=120,n=150,  cats=['Laptops','Desktops','Peripherals','Networking']),
    }
    rows = []
    pid = 1
    for i in range(60):
        base = run_date
        for day_offset in range(30):
            for src, cfg in sources.items():
                cat = rng.choice(cfg['cats'])
                base_price = abs(rng.normal(cfg['mu'], cfg['sigma']))
                jitter = rng.uniform(0.95, 1.05)
                rows.append({
                    'product_id': f'p{pid:04d}',
                    'source': src,
                    'category': cat,
                    'title': f'{cat} product {pid}',
                    'price': round(base_price * jitter, 2),
                    'currency': 'MAD' if 'ma' in src else 'GBP',
                    'rating': round(rng.uniform(1, 5), 1),
                    'availability': rng.choice(['In stock', 'Out of stock'], p=[0.85, 0.15]),
                    'scraped_at': pd.Timestamp(run_date) + pd.Timedelta(days=day_offset, hours=int(rng.integers(0,23))),
                })
        pid += 1
    return pd.DataFrame(rows)


# Load with priority chain
df = pd.DataFrame()
try:
    from bigtable.client import BigtableClient
    client = BigtableClient(
        project_id=os.environ.get('GCP_PROJECT_ID', 'local'),
        instance_id=os.environ.get('BIGTABLE_INSTANCE_ID', 'price-intelligence'),
    )
    rows_bt = list(client._table.read_rows(limit=10000))
    if rows_bt:
        df = pd.DataFrame([client._row_to_dict(r) for r in rows_bt])
        print(f'[Bigtable] {len(df):,} rows')
except Exception as e:
    print(f'Bigtable unavailable: {e}')

if df.empty:
    df = _load_jsonl()
    if not df.empty:
        print(f'[JSONL] {len(df):,} rows')

if df.empty:
    df = _synthetic()
    print(f'[Synthetic demo] {len(df):,} rows')

# ── Clean ────────────────────────────────────────────────────────────────────
df['price'] = pd.to_numeric(df.get('price'), errors='coerce')
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]
if 'scraped_at' in df.columns:
    df['scraped_at'] = pd.to_datetime(df['scraped_at'], errors='coerce', utc=True)
    df['scraped_date'] = df['scraped_at'].dt.date
if 'source' not in df.columns and 'spider' in df.columns:
    df['source'] = df['spider']
if 'category' not in df.columns:
    df['category'] = 'Unknown'

print(f'\nDataset shape: {df.shape}')
print(f'Sources      : {df["source"].unique().tolist()}')
print(f'Date range   : {df["scraped_at"].min()} → {df["scraped_at"].max()}')
df.head(3)

## 2. Central Tendency
Mean, median, mode — per source and overall

In [ ]:
def _mode(s):
    m = s.mode()
    return round(m.iloc[0], 2) if not m.empty else np.nan

central = (
    df.groupby('source')['price']
    .agg(
        n='count',
        mean=lambda s: round(s.mean(), 2),
        median=lambda s: round(s.median(), 2),
        mode=_mode,
        geometric_mean=lambda s: round(np.exp(np.log(s[s>0]).mean()), 2),
    )
    .reset_index()
)

overall_row = pd.DataFrame([{
    'source': '★ OVERALL',
    'n': len(df),
    'mean': round(df['price'].mean(), 2),
    'median': round(df['price'].median(), 2),
    'mode': _mode(df['price']),
    'geometric_mean': round(np.exp(np.log(df['price'][df['price']>0]).mean()), 2),
}])
central = pd.concat([central, overall_row], ignore_index=True)

print('=== Central Tendency ===')
print(central.to_string(index=False))

## 3. Dispersion
Standard deviation, variance, IQR, coefficient of variation

In [ ]:
dispersion = (
    df.groupby('source')['price']
    .agg(
        std=lambda s: round(s.std(), 2),
        variance=lambda s: round(s.var(), 2),
        q25=lambda s: round(s.quantile(0.25), 2),
        q75=lambda s: round(s.quantile(0.75), 2),
        iqr=lambda s: round(s.quantile(0.75) - s.quantile(0.25), 2),
        range_=lambda s: round(s.max() - s.min(), 2),
        cv_pct=lambda s: round(s.std() / s.mean() * 100, 1),   # coefficient of variation
    )
    .reset_index()
)

print('=== Dispersion Metrics ===')
print(dispersion.to_string(index=False))

fig_cv = px.bar(
    dispersion,
    x='source', y='cv_pct', color='source',
    title='Price Volatility — Coefficient of Variation (%) by Source',
    labels={'cv_pct': 'CV (%)', 'source': 'Source'},
    text='cv_pct',
)
fig_cv.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_cv.show()

## 4. Distribution Shape
Skewness, kurtosis, Shapiro-Wilk normality test

In [ ]:
shape_rows = []
for src, grp in df.groupby('source'):
    s = grp['price'].dropna()
    sw_stat, sw_p = stats.shapiro(s.sample(min(len(s), 5000), random_state=42))
    shape_rows.append({
        'source': src,
        'n': len(s),
        'skewness': round(stats.skew(s), 3),
        'kurtosis': round(stats.kurtosis(s), 3),
        'shapiro_W': round(sw_stat, 4),
        'shapiro_p': f'{sw_p:.2e}',
        'is_normal': 'Yes' if sw_p >= 0.05 else 'No',
    })

shape_df = pd.DataFrame(shape_rows)
print('=== Distribution Shape & Normality ===')
print(shape_df.to_string(index=False))
print()
print('Skewness > 0: right-tailed (expensive outliers)')
print('Kurtosis > 0: leptokurtic (heavy tails, more extreme values)')

## 5. Distributions by Category

In [ ]:
# Category summary
cat_summary = (
    df.groupby('category')['price']
    .agg(
        n='count',
        mean=lambda s: round(s.mean(), 2),
        median=lambda s: round(s.median(), 2),
        std=lambda s: round(s.std(), 2),
        q25=lambda s: round(s.quantile(0.25), 2),
        q75=lambda s: round(s.quantile(0.75), 2),
    )
    .sort_values('median', ascending=False)
    .reset_index()
)
print('=== Price by Category ===')
print(cat_summary.to_string(index=False))

top_cats = cat_summary.head(10)['category'].tolist()
fig_box = px.box(
    df[df['category'].isin(top_cats)],
    x='category', y='price', color='source',
    log_y=True,
    title='Price Distribution by Category (log scale, top 10)',
    labels={'price': 'Price (log)', 'category': 'Category'},
)
fig_box.update_layout(xaxis_tickangle=-35)
fig_box.show()

In [ ]:
# Violin plot for source price distributions
fig_violin = px.violin(
    df, x='source', y='price', color='source',
    box=True, points='outliers',
    log_y=True,
    title='Price Violin Plot by Source (log scale)',
    labels={'price': 'Price (log)', 'source': 'Source'},
)
fig_violin.show()

In [ ]:
# Price histogram per source
fig_hist = px.histogram(
    df, x='price', color='source',
    nbins=60, barmode='overlay', opacity=0.65,
    log_x=True, log_y=True,
    title='Price Histogram — All Sources (log-log scale)',
    labels={'price': 'Price (log)', 'count': 'Count (log)'},
)
fig_hist.show()

## 6. Time-Series Analysis
Daily average price per source over the observation window

In [ ]:
if 'scraped_date' in df.columns:
    ts = (
        df.groupby(['scraped_date', 'source'])['price']
        .agg(avg_price='mean', n='count')
        .reset_index()
    )
    ts['avg_price'] = ts['avg_price'].round(2)

    fig_ts = px.line(
        ts, x='scraped_date', y='avg_price', color='source',
        markers=True,
        title='Daily Average Price per Source',
        labels={'avg_price': 'Avg Price', 'scraped_date': 'Date'},
    )
    fig_ts.show()

    # Volume scraped per day
    vol = df.groupby('scraped_date').size().reset_index(name='records')
    fig_vol = px.bar(
        vol, x='scraped_date', y='records',
        title='Daily Scrape Volume',
        labels={'records': 'Records scraped', 'scraped_date': 'Date'},
    )
    fig_vol.show()
else:
    print('scraped_at column not available — skipping time-series plots')

## 7. Price Volatility
Day-over-day percentage change distribution per product

In [ ]:
if 'scraped_at' in df.columns:
    df_s = df.sort_values(['product_id', 'scraped_at'])
    df_s['prev_price'] = df_s.groupby('product_id')['price'].shift(1)
    df_s['pct_change'] = (df_s['price'] - df_s['prev_price']) / df_s['prev_price'] * 100
    df_s = df_s.dropna(subset=['pct_change'])
    df_s = df_s[df_s['pct_change'].abs() < 200]  # remove data errors > 200%

    vol_by_src = (
        df_s.groupby('source')['pct_change']
        .agg(
            mean_chg=lambda x: round(x.mean(), 3),
            std_chg=lambda x: round(x.std(), 3),
            pct_drops=lambda x: round((x < -5).mean() * 100, 1),
            pct_spikes=lambda x: round((x > 5).mean() * 100, 1),
        )
        .reset_index()
    )
    print('=== Price Change Volatility per Source ===')
    print(vol_by_src.to_string(index=False))

    fig_chg = px.histogram(
        df_s, x='pct_change', color='source',
        nbins=80, barmode='overlay', opacity=0.65,
        range_x=[-50, 50],
        title='Distribution of Day-over-Day Price Changes (%)',
        labels={'pct_change': 'Price Change (%)'},
    )
    fig_chg.add_vline(x=0, line_dash='dash', line_color='black')
    fig_chg.add_vline(x=-5, line_dash='dot', line_color='red', annotation_text='-5%')
    fig_chg.add_vline(x=5, line_dash='dot', line_color='green', annotation_text='+5%')
    fig_chg.show()
else:
    print('No scraped_at column — skipping volatility analysis')

## 8. Outlier Detection
IQR fence method and Z-score method

In [ ]:
outlier_rows = []
for src, grp in df.groupby('source'):
    s = grp['price']
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    iqr_outliers = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    z_outliers = (np.abs(stats.zscore(s)) > 3).sum()
    outlier_rows.append({
        'source': src,
        'n': len(s),
        'iqr_outliers': int(iqr_outliers),
        'iqr_pct': round(iqr_outliers / len(s) * 100, 1),
        'z_outliers': int(z_outliers),
        'z_pct': round(z_outliers / len(s) * 100, 1),
        'lower_fence': round(q1 - 1.5 * iqr, 2),
        'upper_fence': round(q3 + 1.5 * iqr, 2),
    })

outlier_df = pd.DataFrame(outlier_rows)
print('=== Outlier Summary ===')
print(outlier_df.to_string(index=False))

## 9. Full Summary Statistics Table

In [ ]:
summary_full = (
    df.groupby(['source', 'category'])['price']
    .agg(
        count='count',
        mean=lambda s: round(s.mean(), 2),
        median=lambda s: round(s.median(), 2),
        std=lambda s: round(s.std(), 2),
        min='min',
        max='max',
        q25=lambda s: round(s.quantile(0.25), 2),
        q75=lambda s: round(s.quantile(0.75), 2),
        skewness=lambda s: round(stats.skew(s), 3),
    )
    .reset_index()
    .sort_values(['source', 'median'], ascending=[True, False])
)
print(f'Full summary: {len(summary_full)} source × category combinations')
summary_full.head(20)

## 10. Heatmap — Median Price by Source × Category

In [ ]:
pivot = df.pivot_table(
    index='source', columns='category', values='price',
    aggfunc='median'
).round(2)

fig_heat = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn_r',
    text=pivot.values.round(0),
    texttemplate='%{text}',
    hovertemplate='%{y} × %{x}<br>Median: %{z:.2f}<extra></extra>',
))
fig_heat.update_layout(
    title='Median Price — Source × Category',
    xaxis_title='Category',
    yaxis_title='Source',
    xaxis_tickangle=-35,
)
fig_heat.show()

## 11. Save HTML Report

In [ ]:
import json

html_path = REPORTS_DIR / '01_descriptive_stats.html'

html_content = f"""
<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='UTF-8'/>
  <title>Descriptive Statistics Report — {run_date}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 40px; }}
    h1 {{ color: #1a1a2e; }}
    h2 {{ color: #16213e; border-bottom: 2px solid #0f3460; padding-bottom:4px; }}
    table {{ border-collapse: collapse; width: 100%; margin-bottom: 24px; }}
    th {{ background: #0f3460; color: white; padding: 8px 12px; }}
    td {{ padding: 6px 12px; border-bottom: 1px solid #ddd; }}
    tr:nth-child(even) {{ background: #f8f9fa; }}
    .meta {{ color: #666; font-size: 0.9em; }}
  </style>
</head>
<body>
<h1>Descriptive Statistics Report</h1>
<p class='meta'>Run date: {run_date} | Records: {len(df):,} | Sources: {df['source'].nunique()}</p>

<h2>Central Tendency</h2>
{central.to_html(index=False, classes='table')}

<h2>Dispersion</h2>
{dispersion.to_html(index=False, classes='table')}

<h2>Distribution Shape &amp; Normality</h2>
{shape_df.to_html(index=False, classes='table')}

<h2>Outlier Detection</h2>
{outlier_df.to_html(index=False, classes='table')}

<h2>Category Summary</h2>
{cat_summary.to_html(index=False, classes='table')}
</body></html>
"""

html_path.write_text(html_content, encoding='utf-8')
print(f'HTML report saved → {html_path}')
print('\n✓ Notebook 1 — Descriptive Statistics complete')